# Weekly S2S anomaly skill (weeks 1-8)

## Purpose

Subseasonal skill of the five initialization configurations following the ESP-Lab S2S protocol (`ESP-Lab/jupyter/s2s_skill`, `esp_lab/diagnostics/s2s_core.py`): weekly lead blocks, drift-corrected anomalies, ACC with significance, and the skill difference between initialization methods.

## Inputs

As `01_compute_and_plot_lead_skill.ipynb`: daily member fields of every configuration and start date, ERA5/IMERG/NOAA OLR references, and the reference climatology of `CLIM_YEARS`.

## Method

`util/fcst_init_skill.py`:

1. Weeks 1-8 = lead days 1-7, ..., 50-56 (7-day means; ensemble mean for the model).
2. Anomalies from the reference calendar-day climatology on the verification days (`CLIM_YEARS`, `SMOOTH_DAYS` running mean; for TREFHT, TS and surface fluxes, whose ERA5 6-hourly files cover only 2011-2012, the ERA5 monthly climatology interpolated to days). With 4 starts per season the hindcast's own lead-dependent climatology is unavailable, so the model drift is removed per week as the mean ensemble-mean error over the other starts of the season (`BIAS = "loo"`, leave-one-out).
3. Pattern ACC (centred, cos-lat weighted) per region, start and week, averaged over starts; MSSS against the climatology forecast.
4. Gridpoint ACC over the start dates of both seasons pooled (8 samples; one-sided t-test critical value 0.62 at 95%, stippled), and its difference from `BASELINE`.

Caveat: ERA5 is the verifying analysis and the source of the HYB_BASE/DA_LND atmospheric initial state, which favours them in week 1.

## Outputs

`data/fcst_init_eval/s2s_weekly/`: `clim_<SEASON>_<VAR>.nc`, `anom_<CONFIG>_<SEASON>_<VAR>.nc` (weekly anomalies), `s2s_weekly_scores.csv` (pattern ACC, MSSS, RMSE per week); figures in `figure/fcst_init_eval/s2s_weekly/`: weekly regional scores, gridpoint ACC maps (2 x 4 weeks) per configuration, ΔACC maps vs `BASELINE`, and the pattern-ACC scorecard.

## 1. Libraries

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

## 2. User setup

Paths and run options. Edit this cell for normal changes; it holds plain values only, so `papermill -p` overrides work.

In [ ]:
INPUT_DATA_ROOT = Path("/global/cfs/cdirs/m4849/zhan391/eam_hindcast")
WORK_DIR = Path("/global/cfs/cdirs/m4849/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026")
OBSERVATION_ROOT = Path("/global/cfs/cdirs/e3sm/zhan391/data")  # reference-data archive
LAND_SUPPORT_DIR = OBSERVATION_ROOT / "lnd_sea_mask"           # land fraction, ELM layer thicknesses
WORKFLOW_NAME = "fcst_init_eval"

# Initialization configurations (configs/fcst_init_experiment_config.py) and seasons (4 starts each)
CONFIGS = ("CLIM_BASE", "HYB_BASE", "DA_ATM", "DA_LND", "DA_COUPLED")
SEASONS = ("DJF", "JJA")
MAX_LEAD = 56          # lead days (8 weeks)
MAX_MEMBERS = None     # e.g. 4 for a quick test
REQUIRE_ALL_MEMBERS = True  # skip a (configuration, season) until every member's stream is regenerated

VARIABLES = ("Z500", "T850", "U850", "U200", "PSL", "TREFHT", "PRECT", "FLUT")
REGIONS = ("GLOBAL", "LAND", "OCEAN", "NH_POLAR", "NINO3.4", "MAR_CONT", "CONUS")
CLIM_YEARS = (2001, 2017)  # ERA5 6-hourly levels end 2017; TREFHT/TS/fluxes use ERA5 monthly
SMOOTH_DAYS = 31
BIAS = "loo"               # "loo" (leave-one-start-out drift removal) or "none"
BASELINE = "HYB_BASE"      # ΔACC and scorecard reference
MAP_VARIABLES = ("TREFHT", "PRECT", "Z500")
PLOT_REGIONS = ("GLOBAL", "LAND")

FORCE_COMPUTE = False
SHOW = True
FIGURE_DPI = 200

## 3. Paths and modules

In [ ]:
repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.fcst_init_experiment_config import MEMBERS, check_configs, init_dates, readiness
from configs.output_paths import workflow_output_dirs

check_configs(CONFIGS)
data_dir, figure_dir = workflow_output_dirs(WORKFLOW_NAME, "s2s_weekly", output_root=DIAGNOSTIC_OUTPUT_ROOT)
ROOTS = dict(data_root=INPUT_DATA_ROOT, land_support_dir=LAND_SUPPORT_DIR)
print(f"[DATA] {data_dir}\n[FIGURE] {figure_dir}")


def members_of(config):
    return MEMBERS[config] if MAX_MEMBERS is None else MEMBERS[config][:MAX_MEMBERS]


def save(fig, name):
    path = figure_dir / name
    fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
    print(f"[FIGURE] {path}")
    if not SHOW:
        plt.close(fig)

## 4. Read input

In [ ]:
status = readiness(CONFIGS, SEASONS, streams=(("atm", "daily"), ("atm", "6hourly")), data_root=INPUT_DATA_ROOT)
summary = status.groupby(["config", "season"]).agg(ready=("ready", "sum"), members=("members", "sum"))
display(summary.unstack("season"))
RUNS = [(c, s) for (c, s), row in summary.iterrows() if row.ready >= (row.members if REQUIRE_ALL_MEMBERS else 1)]
print(f"[RUNS] {len(RUNS)} of {len(summary)} configuration-seasons ready: {RUNS}")

## 5. Derive and save

Reference climatology per season and variable (read once, saved), then the weekly anomalies of each configuration. Saved files are reused unless `FORCE_COMPUTE`.

In [ ]:
from util.fcst_init_data import open_campaign, open_reference, reference_climatology
from util.fcst_init_skill import weekly_anomalies

years = list(range(CLIM_YEARS[0], CLIM_YEARS[1] + 1))
anom_path = lambda c, s, v: data_dir / f"anom_{c}_{s}_{v}.nc"
for season in SEASONS:
    inits = init_dates([season])[season]
    for var in VARIABLES:
        todo = [c for c in CONFIGS if (c, season) in RUNS and (FORCE_COMPUTE or not anom_path(c, season, var).exists())]
        if not todo:
            continue
        clim_path = data_dir / f"clim_{season}_{var}.nc"
        try:
            ref = open_reference(var, inits, max_lead=MAX_LEAD, observation_root=OBSERVATION_ROOT).load()
            if clim_path.exists() and not FORCE_COMPUTE:
                clim = xr.load_dataarray(clim_path)
            else:
                clim = reference_climatology(var, inits, years, max_lead=MAX_LEAD, smooth_days=SMOOTH_DAYS,
                                             observation_root=OBSERVATION_ROOT)
                clim.to_netcdf(clim_path)
                print(f"[WRITE] {clim_path}")
        except FileNotFoundError as err:
            print(f"[SKIP] {var}: {err}")
            continue
        for c in todo:
            model = open_campaign(c, season, var, members=members_of(c), max_lead=MAX_LEAD, **ROOTS).load()
            fa, oa = weekly_anomalies(model, ref, clim, bias=BIAS)
            xr.Dataset({"model": fa, "reference": oa}).assign_attrs(bias=BIAS, clim_years=str(CLIM_YEARS)).to_netcdf(
                anom_path(c, season, var))
            print(f"[WRITE] {anom_path(c, season, var)}")
            del model
anom = {(c, s, v): xr.load_dataset(anom_path(c, s, v)) for c in CONFIGS for s in SEASONS for v in VARIABLES
        if anom_path(c, s, v).exists()}
print(f"[LOAD] {len(anom)} anomaly files")

## 6. Scores

Regional pattern ACC (mean over starts) and MSSS per week; gridpoint ACC over the starts of both seasons and its difference from `BASELINE`.

In [ ]:
from configs.s2s_config import TCC_REGIONS
from util.fcst_init_data import region_masks
from util.fcst_init_skill import critical_acc, gridpoint_acc, msss, pattern_acc, pool_seasons

regions = [r for r in TCC_REGIONS if r["name"] in REGIONS]
masks = None
rows = []
for (c, s, v), ds in anom.items():
    if masks is None:
        masks = region_masks(regions, ds.lat.values, ds.lon.values, land_support_dir=LAND_SUPPORT_DIR)
    pa = pattern_acc(ds.model, ds.reference, masks).mean("init")
    sk = msss(ds.model, ds.reference, masks)
    for r in masks.region.values:
        for w in ds.week.values:
            rows.append(dict(config=c, season=s, variable=v, region=r, week=int(w),
                             pattern_acc=float(pa.sel(region=r, week=w)), msss=float(sk.msss.sel(region=r, week=w)),
                             rmse_anom=float(sk.rmse_anom.sel(region=r, week=w)),
                             rmse_clim=float(sk.rmse_clim.sel(region=r, week=w))))
table = pd.DataFrame(rows)
table.to_csv(data_dir / "s2s_weekly_scores.csv", index=False)
print(f"[WRITE] {data_dir / 's2s_weekly_scores.csv'}")
if len(table):
    display(table[table.region == "GLOBAL"].pivot_table(index=["variable", "config"], columns=["season", "week"],
                                                        values="pattern_acc", sort=False).round(2))

gp_acc = {}
for c in CONFIGS:
    for v in VARIABLES:
        per = {s: anom[c, s, v] for s in SEASONS if (c, s, v) in anom}
        if per:
            fa, oa = pool_seasons({s: d.model for s, d in per.items()}), pool_seasons({s: d.reference for s, d in per.items()})
            gp_acc[c, v] = gridpoint_acc(fa, oa)
n_samples = max((a.attrs["n_samples"] for a in gp_acc.values()), default=0)
R_CRIT = critical_acc(n_samples) if n_samples > 2 else np.nan
print(f"[ACC] gridpoint ACC over {n_samples} starts; 95% one-sided critical value {R_CRIT:.2f}")

## 7. Plot and show

In [ ]:
from util.fcst_init_plotting import plot_config_week_maps, plot_scorecard, plot_week_maps, plot_weekly_regional
from util.fcst_init_skill import relative_change

for s in SEASONS:
    for r in PLOT_REGIONS:
        if len(table[(table.season == s) & (table.region == r)]):
            save(plot_weekly_regional(table, VARIABLES, region=r, season=s), f"s2s_weekly_{s}_{r}.png")

for (c, v), acc in gp_acc.items():
    if v in MAP_VARIABLES:
        save(plot_week_maps(acc, title=f"{c} {v} weekly ACC ({acc.attrs['n_samples']} starts; stippled > {R_CRIT:.2f})",
                            stipple=acc > R_CRIT, label="ACC"), f"s2s_acc_map_{c}_{v}.png")
for v in MAP_VARIABLES:
    if (BASELINE, v) not in gp_acc:
        continue
    diff = {c: gp_acc[c, v] - gp_acc[BASELINE, v] for c in CONFIGS if c != BASELINE and (c, v) in gp_acc}
    for w in (2, 4):
        save(plot_config_week_maps(diff, week=w, title=f"{v} ΔACC vs {BASELINE}, week {w}",
                                   levels=np.linspace(-0.6, 0.6, 13), label="ΔACC"), f"s2s_dacc_{v}_week{w}.png")

for s in SEASONS:
    t = table[(table.season == s) & (table.region == "GLOBAL")]
    if BASELINE not in set(t.config):
        continue
    wide = t.pivot_table(index="config", columns=["variable", "week"], values="pattern_acc", sort=False)
    wide = wide.reindex(index=[c for c in CONFIGS if c in wide.index])
    save(plot_scorecard(relative_change(wide, BASELINE, "pattern_acc"), higher_is_better=True,
                        title=f"{s} global pattern ACC minus {BASELINE}", units="ΔACC"), f"s2s_scorecard_{s}.png")